<a href="https://colab.research.google.com/github/insight4healthlab/course-GS-HLTH-6270/blob/main/notebooks/Assignment_2_text_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 Track A: clinical text
**Read `Assignment_2_Step_by_Step_Guide.docx` before running this notebook.** It contains the setup links, complete workflow, metric explanations, and troubleshooting instructions. Choose this track or the vision track, not both. This guided notebook already contains the technical workflow. Your main tasks are to understand the steps, improve one prompt, run the experiment, and interpret the results. Use only the supplied synthetic notes.

## Student checklist
Run the cells **one at a time, from top to bottom**. Do not use *Run all*.
1. Run the setup and preview 10 notes.
2. Connect securely and test one case.
3. Read the three prompts and make your one improvement to the advanced prompt.
4. Set `RUN_FULL_EXPERIMENT = True` and run the comparison (30 calls, about 3 minutes).
5. Run the supplied evaluation code.
6. Complete the five-case robustness check (5 calls).
7. Review five cases and write your conclusion.
8. Download your files and export the PDF.

Every place where you must write something is marked **Your answer**.

In [ ]:
!pip -q install -U google-genai
import getpass, hashlib, json, os, re, time
import matplotlib.pyplot as plt
import pandas as pd
from google import genai
from google.genai import errors, types

MODEL_NAME = "gemini-3.5-flash-lite"   # Fixed course model. Do not change.
MAX_OUTPUT_TOKENS = 2000
DATA_FILE = "A2_synthetic_notes.csv"
PREDICTIONS_FILE = "A2_text_predictions.csv"
ROBUSTNESS_FILE = "A2_text_robustness.csv"
CACHE_FILE = ".a2_text_cache.json"
FIELDS = ["condition", "medication_action", "followup_interval", "urgency"]

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print(f"Choose {DATA_FILE} from your course package:")
        files.upload()
    except ImportError:
        pass
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"{DATA_FILE} was not found. Upload it with the folder icon on the left, then run this cell again.")

all_notes = pd.read_csv(DATA_FILE)
required = ["note_id", "note_text"] + [f"gold_{f}" for f in FIELDS]
missing = [c for c in required if c not in all_notes.columns]
assert not missing, f"The CSV is missing columns: {missing}. Re-download the course package."
# The same 10 notes for every student: 2 per condition, covering every allowed label.
FIXED_SAMPLE = ["N004", "N011", "N019", "N027", "N033", "N042", "N048", "N056", "N065", "N071"]
notes = all_notes.set_index("note_id").loc[FIXED_SAMPLE].reset_index()
assert len(notes) == 10, "Expected 10 notes. Re-download the course package."
pd.set_option("display.max_colwidth", None)
display(notes[["note_id", "note_text"]])

## 1 Connect securely
Create your Gemini API key at https://aistudio.google.com/apikey (use a personal Google account if your university account is blocked). **In Google Colab:** click the key-shaped **Secrets** icon on the left, add a secret named `GEMINI_API_KEY`, paste your key as its value, and switch on **Notebook access**. Then run the cell below. If the secret is unavailable, the notebook asks you to paste the key into a hidden box. The key is never saved in the notebook. **Do not add billing or payment details** — the free tier is enough for this assignment.

In [ ]:
def get_api_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        key = os.getenv("GEMINI_API_KEY")
    if not key:
        key = getpass.getpass("Paste your Gemini API key (hidden): ").strip()
    if not key:
        raise RuntimeError("No API key was supplied. Ask the instructor for access help.")
    return key

client = genai.Client(api_key=get_api_key())

# Settings are the same for every request, so the prompts are compared fairly.
# Current Gemini models do not accept a custom temperature, and their "thinking"
# counts toward max_output_tokens, so the limit is generous. Do not change these.
GENERATION_CONFIG = types.GenerateContentConfig(
    max_output_tokens=MAX_OUTPUT_TOKENS,
    thinking_config=types.ThinkingConfig(thinking_level="low"),
)
RETRYABLE_CODES = {429, 500, 502, 503, 504}   # busy or rate-limited: worth waiting
SECONDS_BETWEEN_CALLS = 5                      # keeps the notebook under free-tier per-minute limits

def call_with_retry(contents, attempts=4):
    """Send one request. Wait and retry only for temporary errors."""
    for attempt in range(attempts):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME, contents=contents, config=GENERATION_CONFIG)
            return response.text or ""
        except errors.APIError as exc:
            code = getattr(exc, "code", None)
            if code not in RETRYABLE_CODES or attempt == attempts - 1:
                raise
            wait = 20 * (attempt + 1)
            print(f"  Temporary error {code}. Waiting {wait} seconds, then retrying...")
            time.sleep(wait)

# Completed calls are remembered in a small file, so re-running a cell or
# reconnecting after a disconnect never repeats a call that already succeeded.
def _load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE) as handle:
            return json.load(handle)
    return {}

def cached_call(case_id, prompt_name, prompt_text, contents):
    """Return (raw_output, error). Uses the saved answer if this exact request was already made."""
    cache = _load_cache()
    key = f"{MODEL_NAME}|{prompt_name}|{case_id}|" + hashlib.sha1(prompt_text.encode()).hexdigest()
    if key in cache:
        return cache[key], ""
    try:
        raw = call_with_retry(contents)
    except Exception as exc:
        return "", f"{type(exc).__name__}: {exc}"[:300]
    finally:
        time.sleep(SECONDS_BETWEEN_CALLS)
    cache[key] = raw
    with open(CACHE_FILE, "w") as handle:
        json.dump(cache, handle)
    return raw, ""

print(f"Connection setup is ready. Model: {MODEL_NAME}")

## 2 Read and improve the prompts
All three prompts work as written. **Before the full run**, make **one meaningful change** inside `prompt_advanced` (for example: add one invented example, a clearer rule, or a stronger self-check). Leave `prompt_zero` and `prompt_structured` unchanged, and do not change the allowed labels. Then run the cell and explain your change in the answer cell below it.

**How the reference labels were assigned.** Use these rules when you judge errors; you may also use them to improve your prompt.

| Field | Rule |
|---|---|
| condition | The condition assessed and managed at **this** visit, written in full (`HTN` = hypertension, `T2DM` = type 2 diabetes, `OA` = osteoarthritis). Past history, family history and other conditions do not count. |
| medication_action | What happens at this visit to the medication for that condition: `start` (new medication), `continue` (no change), `increase` (dose raised), `stop` (discontinued or put on hold). Other medications and non-drug treatments do not count. |
| followup_interval | The planned clinic review, converted to the nearest allowed label: 7 days = `1 week`, a fortnight = `2 weeks`, 4 weeks = `1 month`, 12 weeks = `3 months`. |
| urgency | `urgent` = same-day assessment is arranged. `soon` = some part of care is deliberately expedited or prioritised within days, but not the same day. `routine` = everything else, including standard advice to call if symptoms worsen. |

In [ ]:
CONDITIONS = "hypertension, type 2 diabetes, asthma, migraine, osteoarthritis"
ACTIONS = "start, continue, increase, stop"
INTERVALS = "1 week, 2 weeks, 1 month, 3 months"
URGENCIES = "routine, soon, urgent"

def prompt_zero(note):
    return (
        "Extract the condition, medication action, follow-up interval, and urgency. "
        "Return JSON with keys condition, medication_action, followup_interval, and urgency.\n"
        f"Note: {note}"
    )

def prompt_structured(note):
    return (
        "Read the synthetic note and return exactly one JSON object.\n"
        f"Allowed condition: {CONDITIONS}\n"
        f"Allowed medication_action: {ACTIONS}\n"
        f"Allowed followup_interval: {INTERVALS}\n"
        f"Allowed urgency: {URGENCIES}\n"
        "Use exactly these keys: condition, medication_action, followup_interval, urgency.\n"
        "Do not add facts that are not stated. Do not include text outside the JSON.\n"
        f"Note: {note}"
    )

def prompt_advanced(note):
    # ---- Make your ONE change inside this function ----
    return (
        "Extract four fields from the synthetic note.\n"
        f"Allowed condition: {CONDITIONS}\n"
        f"Allowed medication_action: {ACTIONS}\n"
        f"Allowed followup_interval: {INTERVALS}\n"
        f"Allowed urgency: {URGENCIES}\n"
        "Before answering, silently check that every value is supported by the note and uses an allowed label.\n"
        "Ignore unrelated information. Return only one JSON object with keys condition, medication_action, "
        "followup_interval, and urgency. Do not provide medical advice.\n"
        f"Note: {note}"
    )

print("Prompts are defined. This is what the advanced prompt looks like for the first note:\n")
print(prompt_advanced(notes.iloc[0].note_text))

**What I changed in `prompt_advanced` and why I expect it to help (2–3 sentences)**

**Your answer:** *(double-click this cell, replace this text, then press Shift+Enter)*

## 3 Test one case
This makes only one API call. You should see something that looks like `{"condition": "...", ...}`. If you see an error instead, use the troubleshooting table in the guide before continuing.

In [ ]:
first = notes.iloc[0]
test_response, test_error = cached_call(first.note_id, "zero", prompt_zero(first.note_text), prompt_zero(first.note_text))
print(test_response if not test_error else f"ERROR - fix this before continuing:\n{test_error}")

## 4 Run the prompt comparison
With `RUN_FULL_EXPERIMENT = False` the cell tries each of the three prompts on **one** note (3 calls) so you can check that everything works. When it does, change `False` to `True` and run the cell again: it processes 10 notes with three prompts, one note at a time (30 calls, about 3 minutes).

Each answer is saved as soon as it arrives. If the cell stops or Colab disconnects, simply run it again — finished calls are not repeated.

If you edit `prompt_advanced` after the full run, run the prompt cell and this cell again: only the 10 advanced-prompt calls are repeated. Please do this at most once or twice.

In [ ]:
def parse_json_object(raw):
    match = re.search(r"\{.*\}", raw or "", flags=re.S)
    if not match:
        return {}
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}
    return parsed if isinstance(parsed, dict) else {}

def clean(value):
    """Lower-case and trim a label so that 'Asthma ' and 'asthma' count as the same answer."""
    if value is None or isinstance(value, (dict, list)):
        return None
    return str(value).strip().lower()

def run_prompt(prompt_name, prompt_function, rows):
    out = []
    for _, row in rows.iterrows():
        prompt_text = prompt_function(row.note_text)
        raw, error = cached_call(row.note_id, prompt_name, prompt_text, prompt_text)
        parsed = parse_json_object(raw)
        record = {"note_id": row.note_id, "prompt": prompt_name}
        for field in FIELDS:
            record[f"gold_{field}"] = clean(row[f"gold_{field}"])
        for field in FIELDS:
            record[f"pred_{field}"] = clean(parsed.get(field))
        record.update({"raw_output": raw, "error": error, "valid_json": bool(parsed)})
        out.append(record)
        print(f"  {prompt_name:<11} {row.note_id}: {'ERROR' if error else 'ok'}")
    return out

prompt_functions = {"zero": prompt_zero, "structured": prompt_structured, "advanced": prompt_advanced}
RUN_FULL_EXPERIMENT = False  # Change to True after the 3-call check works.
evaluation_notes = notes if RUN_FULL_EXPERIMENT else notes.head(1)
results = []
for prompt_name, prompt_function in prompt_functions.items():
    results += run_prompt(prompt_name, prompt_function, evaluation_notes)
    pd.DataFrame(results).to_csv(PREDICTIONS_FILE, index=False)
predictions = pd.DataFrame(results)
n_errors = int((predictions["error"] != "").sum())
print(f"\nSaved {len(predictions)} rows to {PREDICTIONS_FILE}. Calls that failed with an error: {n_errors}")
if n_errors:
    print("Some calls failed. Wait a few minutes and run this cell again; only the failed calls are repeated.")
if not RUN_FULL_EXPERIMENT:
    print("This was only the 3-call check. Set RUN_FULL_EXPERIMENT = True and run this cell again.")
display(predictions.drop(columns="raw_output").head(6))

## 5 Evaluate the three prompts
The code below is provided. Run it; do not rewrite it. Labels are compared after lower-casing and trimming spaces, so `Asthma` counts as `asthma`, but `in 2 weeks` does **not** count as `2 weeks`.

In [ ]:
if not RUN_FULL_EXPERIMENT:
    print("NOTE: these numbers use ONE note only. Set RUN_FULL_EXPERIMENT = True in step 4 for your real results.\n")
metric_rows = []
for prompt_name in prompt_functions:
    group = predictions[predictions["prompt"] == prompt_name]
    correct = pd.DataFrame({f: group[f"pred_{f}"].fillna("<invalid>").eq(group[f"gold_{f}"]) for f in FIELDS})
    metric_rows.append({
        "prompt": prompt_name,
        **{f"{f}_accuracy": correct[f].mean() for f in FIELDS},
        "complete_record_accuracy": correct.all(axis=1).mean(),
        "invalid_output_rate": 1 - group["valid_json"].mean(),
    })
metrics = pd.DataFrame(metric_rows).set_index("prompt")
display(metrics.round(3))
metrics.drop(columns="invalid_output_rate").plot(kind="bar", figsize=(10, 4), ylim=(0, 1.05))
plt.ylabel("Accuracy")
plt.title(f"Prompt comparison on {len(evaluation_notes)} note(s)")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation (4–6 sentences).** Which prompt performed best overall? Which field was hardest? Was invalid JSON a problem? Is the difference large enough to matter with only 10 notes?

**Your answer:** *(double-click this cell, replace this text, then press Shift+Enter)*

## 6 Five-case robustness check
A robust system gives similar answers when the instruction is reworded without changing its meaning. (The model can also vary slightly between identical requests, so interpret one changed answer cautiously.)
1. In the cell below, change the rewording line so the instruction says the same thing in different words.
2. Write your prediction in the answer cell **before** running.
3. Set `RUN_ROBUSTNESS = True` and run. This adds five calls on the first five notes.

**My prediction before running (1 sentence)**

**Your answer:** *(double-click this cell, replace this text, then press Shift+Enter)*

In [ ]:
def prompt_reworded(note):
    # ---- Edit the line below: reword the instruction, keep the same task and allowed labels ----
    return prompt_advanced(note).replace("Extract four fields from the synthetic note.",
                                         "From the synthetic note below, pull out these four items.")

RUN_ROBUSTNESS = False  # Change to True when ready; this adds five calls.
if RUN_ROBUSTNESS and not RUN_FULL_EXPERIMENT:
    print("Finish step 4 with RUN_FULL_EXPERIMENT = True first.")
elif RUN_ROBUSTNESS:
    first_five = notes.head(5)
    assert prompt_reworded(first_five.iloc[0].note_text) != prompt_advanced(first_five.iloc[0].note_text), \
        "Your reworded prompt is identical to the advanced prompt. Edit prompt_reworded first."
    robustness = pd.DataFrame(run_prompt("reworded", prompt_reworded, first_five))
    robustness.to_csv(ROBUSTNESS_FILE, index=False)
    original = predictions[predictions["prompt"] == "advanced"].set_index("note_id")
    reworded = robustness.set_index("note_id")
    comparison = pd.DataFrame({"note_id": first_five["note_id"].values}).set_index("note_id")
    for field in FIELDS:
        comparison[f"{field} | gold"] = reworded[f"gold_{field}"]
        comparison[f"{field} | advanced"] = original[f"pred_{field}"]
        comparison[f"{field} | reworded"] = reworded[f"pred_{field}"]
    same = pd.DataFrame({f: original.loc[comparison.index, f"pred_{f}"].fillna("<invalid>")
                            .eq(reworded[f"pred_{f}"].fillna("<invalid>")) for f in FIELDS})
    display(comparison)
    print(f"Answers that stayed the same after rewording: {int(same.values.sum())} of {same.size}")
    print(f"Saved {ROBUSTNESS_FILE}")
else:
    print("Robustness check not run yet. Set RUN_ROBUSTNESS = True when ready.")

**What happened (2–3 sentences).** Were the outputs stable? Give one concrete example (note ID and field).

**Your answer:** *(double-click this cell, replace this text, then press Shift+Enter)*

## 7 Review five cases
Run the cell to list the rows with the most problems first (invalid output, then most wrong fields). Pick five rows, read the `raw_output`, and complete the table below. If there are fewer than five errors, include correct cases and explain why they succeeded.

In [ ]:
review = predictions.copy()
review["wrong_fields"] = sum(
    (~review[f"pred_{f}"].fillna("<invalid>").eq(review[f"gold_{f}"])).astype(int) for f in FIELDS)
review = review.sort_values(["valid_json", "wrong_fields"], ascending=[True, False])
print(f"Rows with at least one wrong field: {(review['wrong_fields'] > 0).sum()} of {len(review)}")
display(review.head(10))

| Case ID | Prompt | Reference | Prediction | What happened? | Failure type | Possible risk |
|---|---|---|---|---|---|---|
| *N000 (example)* | *zero* | *followup: 2 weeks* | *a fortnight* | *Right meaning, but not an allowed label* | *Wrong label* | *An automated scheduler would not recognise this follow-up* |
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |

Failure types: wrong label, invalid format, omission, unsupported claim, or correct.

## 8 Conclusion and tool-use declaration
In 150–250 words: (1) identify the best prompt and cite one or two metric values, (2) describe one important error or robustness finding, (3) say whether you would trust the system for this limited teaching task, (4) name at least one safeguard, and (5) state that the results do not establish clinical validity.

**Your answer:** *(double-click this cell, replace this text, then press Shift+Enter)*

**Tool-use declaration** (replace the example row; write "None" if you used no tools)

| Tool (AI, coding, translation, editing) | What I used it for | How I checked its output |
|---|---|---|
| *Example: Grammarly* | *Grammar check of my conclusion* | *Re-read every suggested change* |

## 9 Download your files and export the PDF
1. Run the cell below to download your CSV files (Colab storage is temporary).
2. **File → Download → Download .ipynb** to save the executed notebook.
3. **File → Print → Save as PDF** to create the PDF. Check that the PDF shows your outputs and written answers.
4. Check that your API key does not appear anywhere. Submit the .ipynb, the PDF, and both CSV files.

In [ ]:
for name in [PREDICTIONS_FILE, ROBUSTNESS_FILE]:
    if not os.path.exists(name):
        print(f"{name} does not exist yet - complete that step first.")
        continue
    try:
        from google.colab import files
        files.download(name)
    except ImportError:
        print(f"Saved: {name}")